In [1]:
import json_tricks
answer = {}


inputs = json_tricks.load(open('inputs/inputs.json', 'r'))

# Backpropagation Practice

You are given a graph of operations:

<img src="imgs/task_2_4_graph_02.png" width="400" height="300" />

# Task 1

Write function that calculates value of this graph for any given input vector `x` and set of coefficients $b_1, b_2, c_1, c_2$ packed as vector of weights `w`.
In this function also return all the $z$ values.

In [1]:
import numpy as np

## YOU CAN DEFINE ANY FUNCTIONS YOU NEED ##
def sigmoid (x):
    return 1 / (1 + np.exp(-x))

def graph_value(x, w):
    x1, x2 = x[0], x[1]
    b1, b2 = w[0], w[1]
    c1, c2 = w[2], w[3]

    z= {}

    z['z1'] = x1 + b1
    z['z2'] = x2 + b2
    z['z3'] = sigmoid(z['z1'])
    z['z4'] = sigmoid(z['z2'])
    z['z5'] = np.tanh(z['z2'])
    z['z6'] = z['z5'] * c2
    z['z7'] = z['z1'] * z['z4']
    z['z8'] = z['z7'] * c1
    z['z9'] = z['z6'] * z['z3']
    y = z['z9'] + z['z8']
    return y, list(z.values())

In [3]:
answer['graph_value'] = [graph_value(**input) for input in inputs]

# Task 2

In this graph, find all direct derivatives of all the values over $z$ or $x$ or $w$.

For example, if $y = z_8 + z_9$, you need to find only $\partial_{z_9} y$ and $\partial_{z_8} y$ for that operation.

You have to do that for all the intermediate signals.

Write the result in form of dictionary, for example, $\partial_{z_9} y$:

```
['dy_dz9'] = 1
```

In [2]:
def graph_derivatives(x, w):
    res = {}
    y, z = graph_value(x, w)
    z1, z2, z3, z4, z5, z6, z7, z8, z9 = z[0], z[1], z[2], z[3], z[4], z[5], z[6], z[7], z[8]
    b1, b2 = w[0], w[1]
    c1, c2 = w[2], w[3]

    res['dz1_dx1'] = 1.0
    res['dz1_db1'] = 1.0
    res['dz2_dx2'] = 1.0
    res['dz2_db2'] = 1.0

    res['dz3_dz1'] = sigmoid(z1) * (1 - sigmoid(z1))
    res['dz4_dz2'] = sigmoid(z2) * (1 - sigmoid(z2))
    res['dz7_dz4'] = z1
    res['dz7_dz1'] = z4
    res['dz5_dz2'] = 1 / (np.cosh(z2) ** 2)
    res['dz6_dz5'] = c2
    res['dz6_dc2'] = z5
    res['dz8_dz7'] = c1
    res['dz8_dc1'] = z7
    res['dz9_dz3'] = z6
    res['dz9_dz6'] = z3

    res['dy_dz9'] = 1
    res['dy_dz8'] = 1

    return res

In [5]:
answer['graph_derivatives'] = [graph_derivatives(**input) for input in inputs]

# Task 3

Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{c_1} y$
- calculate that derivative

In [6]:
def dy_dc1(x, w):
    selected_derivs = {}
    der = graph_derivatives(x, w)
    selected_derivs['dy_dz8'] = der['dy_dz8']
    selected_derivs['dz8_dc1'] = der['dz8_dc1']

    dy_dc1 = der['dy_dz8'] * der['dz8_dc1']

    return selected_derivs, dy_dc1

In [7]:
answer['dy_dc1'] = [dy_dc1(**input) for input in inputs]

# Task 4
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{c_2} y$
- calculate that derivative

In [8]:
def dy_dc2(x, w):
    selected_derivs = {}
    der = graph_derivatives(x, w)

    selected_derivs['dy_dz9'] = der['dy_dz9']
    selected_derivs['dz9_dz6'] = der['dz9_dz6']
    selected_derivs['dz6_dc2'] = der['dz6_dc2']
    
    dy_dc2 = der['dy_dz9'] * der['dz9_dz6'] * der['dz6_dc2']

    return selected_derivs, dy_dc2

In [9]:
answer['dy_dc2'] = [dy_dc2(**input) for input in inputs]

# Task 5
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{b_1} y$
- calculate that derivative

In [10]:
def dy_db1(x, w):
    selected_derivs = {}
    der = graph_derivatives(x, w)

    selected_derivs['dy_dz8'] = der['dy_dz8']
    selected_derivs['dy_dz9'] = der['dy_dz9']
    selected_derivs['dz8_dz7'] = der['dz8_dz7']
    selected_derivs['dz7_dz1'] = der['dz7_dz1']
    selected_derivs['dz1_db1'] = der['dz1_db1']
    selected_derivs['dz9_dz3'] = der['dz9_dz3']
    selected_derivs['dz3_dz1'] = der['dz3_dz1']

    import math
    # dy_db1 = math.prod(selected_derivs.values())
    dy_db1 = der['dy_dz8'] * der['dz8_dz7'] * der['dz7_dz1'] * der['dz1_db1'] + der['dz9_dz3'] * der['dz3_dz1'] * der['dz1_db1']

    return selected_derivs, dy_db1

In [11]:
answer['dy_db1'] = [dy_db1(**input) for input in inputs]

# Task 6
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{b_2} y$
- calculate that derivative

In [12]:
def dy_db2(x, w):
    selected_derivs = {}
    der = graph_derivatives(x, w)

    selected_derivs['dy_dz8'] = der['dy_dz8']
    selected_derivs['dz8_dz7'] = der['dz8_dz7']
    selected_derivs['dz7_dz4'] = der['dz7_dz4']
    selected_derivs['dz4_dz2'] = der['dz4_dz2']
    selected_derivs['dz2_db2'] = der['dz2_db2']

    selected_derivs['dy_dz9'] = der['dy_dz9']
    selected_derivs['dz9_dz6'] = der['dz9_dz6']
    selected_derivs['dz6_dz5'] = der['dz6_dz5']
    selected_derivs['dz5_dz2'] = der['dz5_dz2']

    dy_db2 = der['dy_dz8'] * der['dz8_dz7'] * der['dz7_dz4'] * der['dz4_dz2'] * der['dz2_db2'] + der['dz9_dz6'] * der['dz6_dz5'] * der['dz5_dz2'] * der['dz2_db2']

    return selected_derivs, dy_db2

In [13]:
answer['dy_db2'] = [dy_db2(**input) for input in inputs]

In [14]:
json_tricks.dump(answer, '.answer.json')

'{"graph_value": [[-0.3709437475832409, [1.1444026911119252, 1.3847655552368408, 0.7584870610795774, 0.7997552788670441, 0.8820139250377154, -0.20651205639513331, 0.9152420933664135, -0.21430702485059627, -0.15663672273264462]], [-0.707721843991169, [1.1097384295724393, 1.3099947727388734, 0.7520803434104762, 0.7875122811662103, 0.864274090196551, -0.40251815904439764, 0.8739326421703995, -0.4049958487081057, -0.30272599528306326]], [0.059224013700513646, [-1.4829555609469987, -2.4755677738987707, 0.1849814142676477, 0.0775888179168375, -0.9859486784088057, -0.3098317422658027, -0.11506076899707829, 0.11653712756985114, -0.0573131138693375]], [0.8461752790658713, [0.5576246934003432, -1.6380800018218271, 0.6359027622340151, 0.16272648639125625, -0.9272036833188647, 1.321031766058989, 0.09074030708203938, 0.00612753003008102, 0.8400477490357903]], [0.6102246564906444, [-1.6953763019474855, 0.48662060805553803, 0.15507011187630465, 0.6193100114629363, 0.45153013859805263, -0.131708519272